# Analyse SAE 2.04 | Partie Mathématiques

Dans le cadre de la SAE 2.04, la deuxième partie consiste à analyser des données issues de deux fichiers CSV représentant les absences d'une promotion de BUT Informatique à Laval.

Cette analyse s'articule autour de plusieurs axes :
- calcul d'indicateurs statistiques
- création de nouvelles variables
- visualisation des résultats

Contraintes imposées :
- traiter chaque type de variable selon les méthodes étudiées en cours
- faire preuve de créativité dans le choix des variables à inventer et à étudier

---

Les deux fichiers CSV sont constitués des colonnes suivantes :

**CSV 1 — Données étudiants**

| Colonne        | Description                                                                                                        |Type        |
|----------------|--------------------------------------------------------------------------------------------------------------------|------------|
| `code_etu`     | Identifiant unique de l'étudiant (fictif)                                                                          |            |
| `nom`          | Nom de l'étudiant (anonymisé)                                                                                      |            |
| `prenom`       | Prénom de l'étudiant (anonymisé)                                                                                   |            |
| `clt_parcsup`  | Classement Parcoursup lors du recrutement — attention, les classements bac général et bac techno sont indépendants |continue    |
| `bac`          | Type de bac obtenu                                                                                                 |nominale    |
| `ue_i_j`       | Note obtenue à l'UE j du semestre i                                                                                |continue    |
| `res_but_1`    | Résultat à l'issue de la 1ère année de BUT                                                                         |ordinale    |
| `res_but_2`    | Résultat à l'issue de la 2ème année de BUT                                                                         |ordinale    |
| `res_but_3`    | Résultat à l'issue de la 3ème année de BUT                                                                         |ordinale    |
| `avis_pe`      | Avis de poursuite d'études                                                                                         |ordinale    |
| `app_but_2`    | Apprentissage en BUT 2                                                                                             |nominale    |
| `app_but_3`    | Apprentissage en BUT 3                                                                                             |nominale    |
| `but_3_etr`    | Année de BUT 3 effectuée à l'étranger                                                                              |nominale    |

**CSV 2 — Données absences**

| Colonne         | Description                                                                                         |Type          |
|-----------------|-----------------------------------------------------------------------------------------------------|--------------|
| `date`          | Date et heure de début de la séance concernée (format datetime)                                     |              |
| `code_etu`      | Identifiant unique de l'étudiant                                                                    |              |
| `nature`        | Nature de l'absence : `absence` ou `retard`                                                         |nominale      |
| `duree_retard`  | Durée du retard, renseignée uniquement si `nature == "retard"`                                      |discret       |
| `justifiee`     | Indique si l'absence est justifiée                                                                  |nominale      |
| `excusee`       | Indique si l'absence est excusée                                                                    |nominale      |
| `enseignant`    | Nom de l'enseignant concerné                                                                        |nominale      |
| `cours`         | Intitulé du cours concerné                                                                          |nominale      |
| `type`          | Type de cours concerné                                                                              |nominale      |
| `duree_cours`   | Durée du cours concerné (minutes)                                                                   |discret       |

---

## Sommaire

1. [Cours sur l'analyse des données](#1-cours-sur-lanalyse-des-données)
   - 1.1 [Variables univariées qualitatives](#11-variables-univariées-qualitatives)
      - 1.1.1 [Nominales](#111-nominales)
      - 1.1.2 [Ordinales](#112-ordinales)
   - 1.2 [Variables univariées quantitatives](#12-variables-univariées-quantitatives)
      - 1.2.1 [Discrètes](#121-discrètes)
      - 1.2.2 [Continues](#122-continues)
   - 1.3 [Variables bivariées](#13-variables-bivariées)
      - 1.3.1 [Quantitative — Quantitative](#131-quantitative--quantitative)
      - 1.3.2 [Quantitative — Qualitative](#132-quantitative--qualitative)
      - 1.3.3 [Qualitative — Qualitative](#133-qualitative--qualitative)
2. [Choix des analyses et variables créées](#2-choix-des-analyses-et-variables-créées)
3. [Préparation des données](#3-préparation-des-données)
   - 3.1 [Chargement des fichiers CSV](#31-chargement-des-fichiers-csv)
   - 3.2 [Nettoyage et traitement](#32-nettoyage-et-traitement)
4. [Analyse statistique](#4-analyse-statistique)
   - 4.1 [Étude des moyennes par UE (`ue_i_j`)](#41-étude-des-moyennes-par-ue)
   - 4.2 [Répartition des absences et des retards (`nature`)](#42-répartition-des-absences-et-des-retards)
   - 4.3 [Analyse de l'assiduité par type de cours (`type`)](#43-analyse-de-lassiduité-par-type-de-cours)
   - 4.4 [Identification des cours les plus désertés (`cours`)](#44-identification-des-cours-les-plus-désertés)
   - 4.5 [Évolution des taux de réussite annuelle (`res_but_1` / `2` / `3`)](#45-évolution-des-taux-de-réussite-annuelle)
   - 4.6 [Analyse de la durée et des extrapolations des retards (`duree_retard`)](#46-analyse-de-la-durée-des-retards)
   - 4.7 [Statistiques des signalements par enseignant (`enseignant`)](#47-statistiques-des-signalements-par-enseignant)
   - 4.8 [Influence du volume d'absences sur la moyenne (`nb_abs_ret` $\times$ `avg_note_per_year`)](#48-influence-du-volume-dabsences-sur-la-moyenne)
   - 4.9 [Comparaison de la réussite selon la filière de bac (`avg_note` $\times$ `type_bac`)](#49-comparaison-de-la-réussite-selon-le-bac)
   - 4.10 [Tendance à l'absentéisme par type de bac (`type_bac` $\times$ `nb_abs_ret`)](#410-tendance-à-labsentéisme-par-type-de-bac)
   - 4.11 [Appétence pour la mobilité internationale selon le bac (`but_3_etr` $\times$ `type_bac`)](#411-mobilité-internationale-selon-le-bac)
   - 4.12 [Orientation vers l'apprentissage selon le bac (`alternance` $\times$ `type_bac`)](#412-orientation-vers-lapprentissage-selon-le-bac)
   - 4.13 [Impact du statut d'alternant sur les performances (`alternance` $\times$ `avg_note`)](#413-impact-du-statut-dalternant-sur-les-performances)
   - 4.14 [Fiabilité des données d'assiduité des alternants (`alternance` $\times$ `nb_abs_ret`)](#414-fiabilité-des-données-des-alternants)
   - 4.15 [Influence du classement d'entrée sur l'avis des professeurs (`clt_parcsup` $\times$ `avis_pe`)](#415-influence-du-classement-sur-lavis-enseignant)
   - 4.16 [Corrélation entre excellence des notes et avis de poursuite (`avis_pe` $\times$ `avg_note_per_year`)](#416-corrélation-entre-notes-et-avis-de-poursuite)
   - 4.17 [Lien entre le classement Parcoursup et les notes obtenues (`avg_note` $\times$ `clt_parcsup`)](#417-lien-entre-classement-parcoursup-et-notes)
5. [Conclusion](#5-conclusion)
---

## Cours sur l'analyse des données

### Variables univariées qualitatives

#### Variables univariées qualitatives nominales

Une variable qualitative nominale prend des modalités comme valeurs, sans ordre logique entre elles.

Les outils d'analyse :

- **Tableau de distribution** (effectifs ou fréquences) : permet de visualiser la répartition dans chaque modalité
- **Mode** : indique la modalité la plus fréquente
- **Représentation graphique** : diagramme en barres, circulaire ou en bâtons pour une lecture plus visuelle de la répartition

---

#### Variables univariées qualitatives ordinales

Une variable qualitative ordinale prend des modalités comme valeurs, avec un ordre logique entre elles.

Les outils d'analyse :

- **Tableau de distribution** (effectifs ou fréquences) : permet de visualiser la répartition dans chaque modalité
- **Mode** : indique la modalité la plus fréquente
- **Médiane ordinale** : indique la modalité centrale en tenant compte de l'ordre
- **Représentation graphique** : diagramme en barres ou en bâtons, en respectant l'ordre des modalités

---

### Variables univariées quantitatives

#### Variables univariées quantitatives discrètes

Une variable quantitative discrète prend des valeurs numériques isolées (entières ou dénombrables).

Les outils d'analyse :

- **Tableau de distribution** (effectifs ou fréquences) : permet de visualiser la répartition des valeurs
- **Représentation graphique** : diagramme en bâtons (en respectant l'échelle), courbe des fréquences cumulées
- **Résumés de position** : moyenne et médiane, permettant de caractériser un individu type
- **Résumés de dispersion** : variance et écart-type, indiquant l'étendue des écarts autour de la moyenne
- **Boîte à moustaches** : synthèse visuelle des extremums, quartiles et médiane

---

#### Variables univariées quantitatives continues

Une variable quantitative continue prend ses valeurs dans des intervalles.

Les outils d'analyse :

- **Tableau de distribution** (effectifs, fréquences ou densités) : permet de visualiser la répartition par classe
- **Représentation graphique** : histogramme des densités, courbe des fréquences cumulées
- **Résumés de position** : moyenne pour l'individu type, médiane pour limiter l'effet des valeurs extrêmes
- **Résumés de dispersion** : variance, écart-type et quartiles permettant de détecter les valeurs aberrantes

---

### Variables bivariées

#### Variables bivariées quantitative - quantitative

Pour analyser la relation entre deux variables quantitatives :

- **Nuage de points** : représentation graphique de la distribution jointe des deux variables
- **Covariance** : mesure le sens de la relation linéaire entre les deux variables
- **Coefficient de corrélation de Pearson** : mesure l'intensité et le sens de la liaison linéaire, compris entre -1 et 1
- **Droite de régression linéaire** : modélise la relation entre les deux variables et permet de faire des prédictions

---

#### Variables bivariées quantitative - qualitative

Pour analyser la relation entre une variable quantitative et une variable qualitative :

- **Tableau des moyennes par modalité** : compare la moyenne de la variable quantitative pour chaque groupe
- **Boîtes à moustaches par groupe** : visualise et compare la distribution de la variable quantitative selon les modalités
- **Rapport de corrélation η²** : mesure la part de variance de la variable quantitative expliquée par la variable qualitative

---

#### Variables bivariées qualitative - qualitative

Pour analyser la relation entre deux variables qualitatives :

- **Tableau de contingence** : représente la distribution jointe des effectifs selon les deux variables
- **Distributions marginales** : permettent d'analyser la distribution de chaque variable indépendamment
- **Distributions conditionnelles** : permettent d'analyser la distribution d'une variable selon les modalités de l'autre
- **Test du Chi²** : mesure si les deux variables sont indépendantes ou liées

### 1. Tableau des Variables Construites (Feature Engineering)

Ce tableau présente les nouvelles variables générées et calculées à partir des données brutes pour enrichir l'analyse.

| Nom de la variable | Source / Mode de calcul | Description / Objectif |
| :--- | :--- | :--- |
| **`nb_ret`** | Compte des lignes par étudiant dans les fichiers d'absences | Nombre total d'absences accumulés par un élève. |
| **`nb_abs`** | Compte des lignes par étudiant des retards | Nombre total de retards accumulés par un élève. |
| **`avg_ue_i_per_year`** | Moyenne des colonnes `ue_i_j` regroupées par année | Moyenne obtenue pour une UE spécifique par élève et par année. |
| **`avg_note_per_year`** | Moyenne de l'ensemble des UE d'une même année | Moyenne générale annuelle calculée pour chaque élève. |
| **`avg_note`** | Calculée à partir des `avg_note_per_year` | Moyenne générale globale de l'étudiant sur l'ensemble de son cursus. |
| **`alternance`** | Basée sur `app_but_2` ou `app_but_3` | Variable binaire indiquant si l'élève est apprenti/alternant au cours de son BUT. |
| **`est_defaillant`** | Algorithme basé sur l'INE, les dates et la durée des absences | Détermine si un élève est déclaré défaillant selon son nombre de demi-journées d'absences injustifiées. |

---

### 2. Tableau des Analyses et Croisements d'Étude

Ce tableau synthétise les axes d'analyse (univariés et bivariés) permettant de répondre aux problématiques de la SAE.

| Variables étudiées | Problématique / Objectif de l'analyse |
| :--- | :--- |
| **`ue_i_j`** | Analyser les moyennes de chaque UE pour identifier d'éventuels écarts de notation ou difficultés majeures. |
| **`nature`** | Déterminer s'il y a globalement une majorité d'absences ou de retards. |
| **`type`** | Identifier la typologie de cours (CM, TD, TP) qui comptabilise le plus de manquements. |
| **`cours`** | Découvrir quelle matière ou quel cours précis enregistre le plus fort taux d'absentéisme. |
| **`res_but_1` / `2` / `3`** | Analyser les taux de réussite, d'admission et de passage d'une année à l'autre. |
| **`duree_retard`** | Calculer la durée moyenne d'un retard et repérer les valeurs aberrantes (extrapolées). |
| **`enseignant`** | Identifier quel enseignant déclare ou fait face au plus grand nombre d'absences/retards. |
| **`nb_abs_ret` $\times$ `avg_note_per_year`** | Mesurer si le volume d'absences et de retards influence négativement la moyenne de l'élève. |
| **`avg_note` $\times$ `type_bac`** | Déterminer quelle filière de baccalauréat obtient le meilleur taux de réussite académique. |
| **`type_bac` $\times$ `nb_abs_ret`** | Observer si le profil du bac d'origine est lié à une tendance à l'absentéisme. |
| **`but_3_etr` $\times$ `type_bac`** | Identifier quel type de bac s'oriente le plus vers une mobilité internationale en BUT 3. |
| **`alternance` $\times$ `type_bac`** | Analyser s'il y a un type de bac qui privilégie ou accède le plus à la voie de l'apprentissage. |
| **`alternance` $\times$ `avg_note`** | Étudier si le statut d'alternant impacte positivement ou négativement l'investissement et les notes scolaires. |
| **`alternance` $\times$ `nb_abs_ret`** | Vérifier la fiabilité et la cohérence des données d'assiduité spécifiques aux alternants. |
| **`clt_parcsup` $\times$ `avis_pe`** | Analyser si le classement initial sur Parcoursup a une influence sur l'avis final de poursuite d'études. |
| **`avis_pe` $\times$ `avg_note_per_year`** | Vérifier si l'avis de poursuite d'études est directement corrélé à l'excellence des notes obtenues. |
| **`avg_note` $\times$ `clt_parcsup`** | Évaluer l'efficacité de Parcoursup : les étudiants les mieux classés à l'entrée sont-ils ceux qui ont les meilleures moyennes ? |